In [8]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
import joblib
import importlib.util
import utils.ovo_svm as ovo_svm
sys.modules["ovo_svm"] = ovo_svm
from utils.ovo_svm import OvO_SVM
from sklearn.preprocessing import LabelEncoder
from utils.Hog import HoG, calc_gradients, predict_char
from utils.MSER import merge_boxes, filter_char_boxes, merge_char_words, get_word_boxes, sort_boxes_reading_order, sort_word_chars, remove_image_border_box, remove_holes, remove_large_boxes
from utils.text_detector import extract_word_images
import easyocr

In [7]:
def recognize_word(word_img, mser, model, le):
    gray = cv2.cvtColor(word_img, cv2.COLOR_BGR2GRAY)
    _, boxes = mser.detectRegions(gray)

    unique_boxes = list({tuple(b) for b in boxes})
    unique_boxes = merge_boxes(unique_boxes, threshold=0.3)
    unique_boxes = sort_word_chars(unique_boxes)
    unique_boxes = remove_image_border_box(unique_boxes, word_img.shape)
    unique_boxes = remove_holes(unique_boxes)
    unique_boxes = remove_large_boxes(unique_boxes, word_img.shape)

    text = ""
    char_confidences = []
    for box in unique_boxes:
        x, y, w, h = box
        char_img = gray[y:y+h, x:x+w]
        if char_img.size == 0:
            continue

        padded = cv2.copyMakeBorder(char_img, 5, 5, 5, 5, cv2.BORDER_CONSTANT, value=255)
        blurred = cv2.GaussianBlur(padded, (5, 5), 0)
        binarized = cv2.adaptiveThreshold(
            blurred, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 11, 2
        )
        resized = cv2.resize(binarized, (128, 128), interpolation=cv2.INTER_CUBIC)
        dilated = cv2.dilate(resized, np.ones((3, 3), np.uint8), iterations=1)
        normalized = dilated.astype(np.float32) / 255.0

        magnitudes, orientations = calc_gradients(normalized)
        features = HoG(orientations, magnitudes)
        predicted_label, confidence = predict_char(features, model, le)
        text += le.inverse_transform(predicted_label)[0]
        if isinstance(confidence, (np.ndarray, list)):
            confidence = confidence[0]
        char_confidences.append(float(confidence))

    word_text = text.strip().lower()
    word_confidence = float(np.mean(char_confidences)) if char_confidences else 0.0
    return word_text, word_confidence


In [10]:
mser = cv2.MSER_create()
model = OvO_SVM().load("./models/from_scratch_SVM")
le = joblib.load("./models/from_scratch_SVM/OvO_SVM_label_encoder.joblib")
net = cv2.dnn.readNet("./models/frozen_east_text_detection.pb")

c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\vidseek-jM3xzebT-py3.12\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [5]:
# import warnings
# warnings.filterwarnings("ignore", message=".*pin_memory.*")

In [6]:
# image = cv2.imread("../data/OCR/coco_text/78.jpg")
# word_images = extract_word_images(image, net)
# print(len(word_images))
# reader = easyocr.Reader(['en'], gpu=True)
# results = reader.readtext('../data/OCR/coco_text/58.jpg')

In [3]:
images_folder = "../data/OCR/mjsynth_subset_5k/"
mser_dict = {}
easyocr_dict = {}

In [4]:
from utils.word_distance import get_edit_distance

In [23]:
count = 0
total_confidence = 0.0
total_edit_distances_count = 0
for i, image_name in enumerate(os.listdir(images_folder)):
    image_path = os.path.join(images_folder, image_name)
    img = cv2.imread(image_path)
    word_text = recognize_word(img, mser, model, le)
    label = image_name.split("_")[1]
    # print(label)
    edit_distance_count, confidence = get_edit_distance(label, word_text[0])
    count += 1
    total_confidence += confidence
    total_edit_distances_count += edit_distance_count
    if count % 10 == 0:
        print(f"Processed {count} images. Average confidence: {total_confidence / count}")
        

Processed 10 images. Average confidence: 0.794440836940837
Processed 20 images. Average confidence: 0.7503950216450217
Processed 30 images. Average confidence: 0.6848388648388649
Processed 40 images. Average confidence: 0.6585299422799424
Processed 50 images. Average confidence: 0.6445779220779222
Processed 60 images. Average confidence: 0.6684499759499758
Processed 70 images. Average confidence: 0.6855421562564417
Processed 80 images. Average confidence: 0.6842987914862911
Processed 90 images. Average confidence: 0.6794737053070382
Processed 100 images. Average confidence: 0.6848239538239533
Processed 110 images. Average confidence: 0.6802043158861337
Processed 120 images. Average confidence: 0.6680245911495909
Processed 130 images. Average confidence: 0.6640498390498389
Processed 140 images. Average confidence: 0.6622129457843742
Processed 150 images. Average confidence: 0.6548246753246753
Processed 160 images. Average confidence: 0.6519757846320345
Processed 170 images. Average conf

KeyboardInterrupt: 

In [24]:
avg_total_confidence = total_confidence / count
avg_edit_distance = total_edit_distances_count / count
print(f"Average confidence: {avg_total_confidence}")
print(f"Average edit distance: {avg_edit_distance}")

Average confidence: 0.7119867619393498
Average edit distance: 2.2699137493658044


In [5]:
predicted = []
labels = []

In [11]:
count = 0
total_confidence = 0.0
total_edit_distances_count = 0
for i, image_name in enumerate(os.listdir(images_folder)):
    if(count >= 100):
        break
    image_path = os.path.join(images_folder, image_name)
    img = cv2.imread(image_path)
    word_text = recognize_word(img, mser, model, le)
    label = image_name.split("_")[1]
    # print(label)
    edit_distance_count, confidence = get_edit_distance(label, word_text[0])
    count += 1
    total_confidence += confidence
    total_edit_distances_count += edit_distance_count
    labels.append(label)
    predicted.append(word_text[0])
    if count % 10 == 0:
        print(f"Processed {count} images. Average confidence: {total_confidence / count}")
        

Processed 10 images. Average confidence: 0.794440836940837
Processed 20 images. Average confidence: 0.7503950216450217
Processed 30 images. Average confidence: 0.6848388648388649
Processed 40 images. Average confidence: 0.6585299422799424
Processed 50 images. Average confidence: 0.6445779220779222
Processed 60 images. Average confidence: 0.6684499759499758


KeyboardInterrupt: 

In [13]:
for i in range(len(labels)):
    print(f"Label: {labels[i]}, Predicted: {predicted[i]}")

Label: bayonet, Predicted: e
Label: Cecum, Predicted: nannn
Label: Daycare, Predicted: ehn
Label: Encouraged, Predicted: mauagea
Label: HEREFORDS, Predicted: meheuen
Label: LOCATING, Predicted: nnunm
Label: Meowing, Predicted: mmmm
Label: one3one, Predicted: qnebone
Label: opportunity, Predicted: dd
Label: topmast, Predicted: naenn
Label: UNSUCCESSFULLY, Predicted: ummnrebennn
Label: CARING, Predicted: u
Label: cottons, Predicted: nnnnnne
Label: DUELISTS, Predicted: nnenune
Label: EGGPLANT, Predicted: eeeenen
Label: lionized, Predicted: 
Label: MAPMAKER, Predicted: nnennnnn
Label: NECKLINES, Predicted: nannmmeu
Label: PURIFIED, Predicted: nureeeen
Label: scoopfuls, Predicted: auaannaq
Label: Travailing, Predicted: mnd
Label: xmases, Predicted: u
Label: adversities, Predicted: aena
Label: Britannic, Predicted: 
Label: demesne, Predicted: aenenna
Label: Fess, Predicted: e
Label: IMPULSED, Predicted: nmenmsed
Label: Motorists, Predicted: ew
Label: nikolai, Predicted: nnunn
Label: ogres, P